In [ ]:
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from star_sharp import StarSharp

In [ ]:
ssh = StarSharp(
    "r",
    use_dof="0-16,30-34",
    nkeep=12,
    ortho_transverse=False,
    tqdm=tqdm
)

In [ ]:
U, S, Vh = ssh._svd

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
bkr = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "bkr", ["blue", "black", "red"]
)
axs[0].imshow(Vh.T, origin="lower", aspect="auto", cmap=bkr)
m2_hex_idx = np.where(ssh.use_dof < 5)[0]
cam_hex_idx = np.where((ssh.use_dof >= 5) & (ssh.use_dof < 10))[0]
m1m3_bend_idx = np.where((ssh.use_dof >= 10) & (ssh.use_dof < 30))[0]
m2_bend_idx = np.where(ssh.use_dof >= 30)[0]
if len(m2_hex_idx) > 0:
    if any(len(block) > 0 for block in [cam_hex_idx, m1m3_bend_idx, m2_bend_idx]):
        axs[0].axhline(m2_hex_idx[-1] + 0.5, color="w", alpha=0.2)
if len(cam_hex_idx) > 0:
    if any(len(block) > 0 for block in [m1m3_bend_idx, m2_bend_idx]):
        axs[0].axhline(cam_hex_idx[-1] + 0.5, color="w", alpha=0.2)
if len(m1m3_bend_idx) > 0:
    if len(m2_bend_idx) > 0:
        axs[0].axhline(m1m3_bend_idx[-1] + 0.5, color="w", alpha=0.2)

dof_labels = []
for dof in ssh.use_dof:
    if dof < 5:
        dof_labels.append(["M2 dz", "M2 dx", "M2 dy", "M2 rx", "M2 ry"][dof])
    elif dof < 10:
        dof_labels.append(["Cam dz", "Cam dx", "Cam dy", "Cam rx", "Cam ry"][dof - 5])
    elif dof < 30:
        dof_labels.append(f"M1M3 B{dof - 9}")
    else:
        dof_labels.append(f"M2 B{dof - 29}")
axs[0].set_yticks(np.arange(len(ssh.use_dof)))
axs[0].set_yticklabels(dof_labels)

nkeep = ssh.nkeep if ssh.nkeep is not None else len(S)
axs[0].axvspan(nkeep - 0.5, len(S) - 0.5, color="w", alpha=0.2)
axs[0].set_xlim(-0.5, len(S) - 0.5)
axs[0].set_xticks([i - 1 for i in range(5, len(S) + 1, 5)])
axs[0].set_xticklabels([f"{i}" for i in range(5, len(S) + 1, 5)])

axs[1].plot(np.arange(1, len(S) + 1), S)
axs[1].set_xlim(0.5, len(S) + 0.5)
axs[1].set_xticks([i for i in range(5, len(ssh.use_dof) + 1, 5)])
axs[1].set_xticklabels([f"{i}" for i in range(5, len(ssh.use_dof) + 1, 5)])
axs[1].set_yscale("log")
axs[1].set_yticks([])
axs[1].axvspan(nkeep + 0.5, len(S) + 0.5, color="w", alpha=0.2)

axs[2].plot(np.arange(1, len(S) + 1), np.cumsum(S)/np.sum(S))
axs[2].set_xlim(0.5, len(S) + 0.5)
axs[2].set_xticks([i for i in range(5, len(ssh.use_dof) + 1, 5)])
axs[2].set_xticklabels([f"{i}" for i in range(5, len(ssh.use_dof) + 1, 5)])
axs[2].axvspan(nkeep + 0.5, len(S) + 0.5, color="w", alpha=0.2)

for ax in axs[1:]:
    ax.grid(axis="x", color="w", alpha=0.35, linewidth=0.6)
    ax.set_xlabel("Mode number")
axs[0].set_xlabel("Mode number")
axs[0].set_xticks(np.arange(-0.5, len(S), 1), minor=True)
axs[0].set_yticks(np.arange(-0.5, len(ssh.use_dof), 1), minor=True)
axs[0].grid(which="minor", color="w", alpha=0.15, linewidth=0.4)
axs[0].tick_params(axis="x", which="minor", bottom=False)
axs[0].tick_params(axis="y", which="minor", left=False)
axs[0].set_ylabel("Mode mixing coefficients")
axs[1].set_ylabel("Singular value")
axs[2].set_ylabel("Cumulative sum")



plt.show()